# Project 4: **Build a Deep Research System**
Welcome to project 4! For this project, we shift our focus from tool use and agents to *reasoning* models. You will practice state‑of‑the‑art inference‑time scaling methods such as *Chain‑of‑Thought* prompting and *Tree‑of‑Thoughts*, and briefly explore high-level concepts of training reasoning models using techniques like **STaR**.


Finally, you will put everything together to build a *deep research agent* that can browse the web, reason over what it finds, and give structured answers.

## Learning Objectives  
* Apply common inference‑time scaling methods: **zero‑shot / few‑shot CoT, self‑consistency, sequential revision, tree‑of‑thoughts**  
* Gain intuition for **training** reasoning‑capable models following **STaR** approach 
* Build a minimal **deep‑research agent** that combines step‑by‑step reasoning with live web search   
* Practice extending deep-search to a multi-agent system 

## Roadmap  
0. Environment setup  
1. Inference‑time scaling  
  1.1 Few‑shot.   
  1.2 Zero‑shot CoT.   
  1.3 Self‑consistency.   
  1.4 Sequential revisions.     
  1.5 Tree‑of‑Thought (ToT)
2. Training reasoning models and inspecting deepseek-r1 
3. Deep-research agent  
4. (Optional) Multi-agent deep-research

# 0- Environment setup

### Step 1: Create your environment and install dependencies 
Before we start coding, you need a reproducible setup. Open a terminal in the same directory as this notebook, and use Conda or uv to install the project dependencies.

#### Option 1: Conda
```bash
# Create and activate the conda environment
conda env create -f environment.yaml && conda activate deep_research
```

#### Option 2: uv (Fast alternative)
If you prefer [uv](https://docs.astral.sh/uv/) over Conda:

```bash
# Install uv (skip if already installed)
curl -LsSf https://astral.sh/uv/install.sh | sh

# Create a virtual environment and install dependencies
uv venv .venv-deep-research && source .venv-deep-research/bin/activate
uv pip install -r requirements.txt
```

### Step 2: Register this environment as a Jupyter kernel
```bash
python -m ipykernel install --user --name=deep_research --display-name "deep_research"
```
Now open your notebook and switch to the `deep_research` kernel (Kernel → Change Kernel).

### Step 3: Setup and run Ollama serve

In this project we use the `llama3.2:3b`, `qwen2.5:3b-instruct` and `deepseek-r1:1.5b` models. You can try other smaller or larger reasoning LLMs such as `phi4-mini` to compare performance. Explore available models here: https://ollama.com/library.

Open terminal and run ollama:
```bash
ollama serve
```
Then open another terminal and pull required models: 
```bash
ollama pull llama3.2:3b
ollama pull deepseek-r1:1.5b
ollama pull qwen2.5:3b-instruct
# Additional small reasoning models to compare
# ollama pull phi4-mini
```

**Setup checklist (run these in a terminal from the `project_4` directory):**

| Step | Command |
|------|---------|
| **1. Create env (UV)** | On Apple Silicon: `uv python install 3.12-macos-aarch64` then `uv venv .venv-deep-research --python 3.12-macos-aarch64`, `source .venv-deep-research/bin/activate`, and `uv pip install -r requirements.txt`. On other platforms use `uv venv .venv-deep-research` and the same pip install. |
| **2. Register kernel** | `python -m ipykernel install --user --name=deep_research --display-name "deep_research"` then pick the **deep_research** kernel in the notebook. |
| **3. Ollama** | In a separate terminal: `ollama serve`. Then pull models: `ollama pull llama3.2:3b` etc. |

In [1]:
# Environment check: run this cell to verify your setup (deep_research kernel + deps)
import sys
print("Python:", sys.executable)
try:
    from openai import OpenAI
    print("openai: OK")
except ImportError as e:
    print("openai:", e)
try:
    import langchain
    print("langchain: OK")
except ImportError as e:
    print("langchain:", e)
try:
    import torch
    print("torch:", torch.__version__)
except ImportError as e:
    print("torch: not installed")
try:
    import transformers
    print("transformers:", transformers.__version__)
except ImportError as e:
    print("transformers: not installed")

Python: /Users/tempUser/Projects/ai-engineering-course/project_4/.venv-deep-research/bin/python
openai: OK
langchain: OK
torch: 2.10.0
transformers: 5.0.0


---  
# 1‑ Inference‑time scaling

Inference-time scaling refers to techniques that make an existing model reason better without retraining it. Instead of changing the model’s weights, we achieve reasoning capability by adjusting how we prompt, sample, or aggregate LLM's outputs.

In this section, we’ll explore several inference-time strategies that improve reasoning quality using a non-reasoning base model. You will experiment with and compare methods such as:

- Few-shot Chain-of-Thought (CoT)
- Zero-shot CoT
- Self-consistency
- Sequential revision
- Tree-of-Thoughts (ToT)

### 1.1: Few-Shot CoT

Few-shot prompting provides examples before asking a new question. The model learns from the pattern and applies it to new inputs.

We'll explore this with two models to understand how few-shot interacts with model capabilities:

1. **GPT-2** (no instruction tuning): Doesn't reason by default. We'll see if few-shot examples can elicit reasoning.
2. **Llama 3.2** (instruction-tuned): Already reasons naturally. We'll use few-shot to control the output format.

#### GPT-2: Can few-shot examples elicit reasoning?

GPT-2 is a base language model that just predicts the next token. It wasn't trained to follow instructions or reason step-by-step. Let's see what happens with and without few-shot examples.

In [3]:
import os
import torch
from transformers import pipeline

question = "A rectangle has a perimeter of 36 cm. If the length is twice the width, what is the area?"

# Step 1: Load GPT-2 text-generation from huggingface (use device= to avoid requiring accelerate)
pipe = pipeline("text-generation", model="gpt2", dtype=torch.float32, device="cpu")

# Step 2: Few-shot reasoning examples (short steps + final answer)
few_shot_examples = """Q: A square has perimeter 20 cm. What is its area?
Steps: perimeter=20 so side=20/4=5 cm. Area=side^2=25. Answer: 25 cm²

Q: A rectangle has perimeter 24 cm and length 8 cm. What is the width?
Steps: 2*(8+w)=24, 8+w=12, w=4. Answer: 4 cm

Q: """

# Step 3: Prompt with and without few-shot
prompt_with_fewshot = few_shot_examples + question + "\nSteps:"
prompt_no_fewshot = question + "\n"

# Step 4: Generate and compare
out_no_fewshot = pipe(prompt_no_fewshot, max_new_tokens=80, do_sample=True, pad_token_id=pipe.tokenizer.eos_token_id)[0]["generated_text"]
out_with_fewshot = pipe(prompt_with_fewshot, max_new_tokens=80, do_sample=True, pad_token_id=pipe.tokenizer.eos_token_id)[0]["generated_text"]

print("=== Without few-shot ===")
print(out_no_fewshot)
print("\n=== With few-shot ===")
print(out_with_fewshot)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Passing `generation_config` together with generation-related arguments=({'do_sample', 'pad_token_id', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=80) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=80) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/do

=== Without few-shot ===
A rectangle has a perimeter of 36 cm. If the length is twice the width, what is the area?

The width of the rectangle is determined by the height of the rectangle, and is equal to or less than the height of the rectangle.

This rectangle has the following properties:

The width of the rectangle is determined by the height of the rectangle.

The width of the rectangle is determined by the height of the rectangle. The rectangle has an area of 6 x 4 x 4

=== With few-shot ===
Q: A square has perimeter 20 cm. What is its area?
Steps: perimeter=20 so side=20/4=5 cm. Area=side^2=25. Answer: 25 cm²

Q: A rectangle has perimeter 24 cm and length 8 cm. What is the width?
Steps: 2*(8+w)=24, 8+w=12, w=4. Answer: 4 cm

Q: A rectangle has a perimeter of 36 cm. If the length is twice the width, what is the area?
Steps: 2*(36+w)=36, 2*(8+w)=36, 2*(8+w)=36

Q: A square has perimeter 32 cm and length 7 cm. What is the area?

Steps: 2*(32+w)=32, 2*(8+w)=32, 2*(32+w)=32

Q


#### Llama 3.2: Using few-shot to control output format

Unlike GPT-2, Llama 3.2 is instruction-tuned and already produces reasoning traces by default. So what's the point of few-shot examples?

**The power of few-shot with instruction-tuned models is controlling the output format.** We can make the model follow a specific structure like `[GIVEN]/[FIND]/[SOLVE]/[ANSWER]` that it wouldn't use naturally.

In [4]:
from openai import OpenAI

question = "A rectangle has a perimeter of 36 cm. If the length is twice the width, what is the area?"

# Step 1: Ollama client
client = OpenAI(api_key="ollama", base_url="http://localhost:11434/v1")

# Step 2 & 3: Few-shot examples + new question
few_shot_prompt = """[GIVEN] Square perimeter 20 cm. [FIND] Area. [SOLVE] Side=20/4=5. Area=5*5=25. [ANSWER] 25 cm²

[GIVEN] Rectangle perimeter 24, length 8. [FIND] Width. [SOLVE] 2(8+w)=24, w=4. [ANSWER] 4 cm

[GIVEN] """ + question + " [FIND] Area. [SOLVE] "

# Without few-shot: plain question
resp_no_fewshot = client.chat.completions.create(model="llama3.2:3b", messages=[{"role": "user", "content": question}], max_tokens=150)
resp_with_fewshot = client.chat.completions.create(model="llama3.2:3b", messages=[{"role": "user", "content": few_shot_prompt}], max_tokens=150)

print("=== Without few-shot ===")
print(resp_no_fewshot.choices[0].message.content)
print("\n=== With few-shot (controlled format) ===")
print(resp_with_fewshot.choices[0].message.content)

=== Without few-shot ===
To find the area, we first need to determine the dimensions of the rectangle. Let's call the width "w" and the length "2w", since it's twice the width.

The perimeter of a rectangle is given by:

Perimeter = 2(length + width)

In this case, the perimeter is 36 cm. So we can set up an equation:

36 = 2(2w + w)
36 = 6w

Now, divide both sides by 6 to solve for w:

w = 36/6
w = 6

So, the width is 6 cm.

Since the length is twice the width, the length is:

Length = 2w = 2(6) = 

=== With few-shot (controlled format) ===
Let's solve it step by step.

[Given] A rectangle has a perimeter of 36 cm. 

 Let's denote the width as W and the length as L. We know that:

 Perimeter = 2(Length + Width)

We are given that the length is twice the width, so we can write an equation based on this relationship:

2L = L/2

To simplify, let's multiply both sides by 4 to get rid of the fraction:

4L = L

Now, since perimeter is equal to 36 cm, we can set up another equation using the

### 1.2: Zero‑Shot Chain‑of‑Thought
Zero-shot CoT encourages the model to reason without examples by adding a short cue such as “Let’s think step by step.” This simple phrase often activates the model’s latent reasoning ability even when no demonstrations are provided. It serves as a baseline to compare with few-shot and other inference-time scaling methods.

In [5]:
from openai import OpenAI

# Step 1: Ollama client
client = OpenAI(api_key="ollama", base_url="http://localhost:11434/v1")

# Step 2 & 3: Question + zero-shot CoT cue
question = "If 3 pencils cost $1.20, how much do 5 pencils cost?"
prompt = f"You are a careful reasoner. Answer the following question.\n\n{question}\n\nLet's think step by step."

# Step 4: Call model and print chain + answer
resp = client.chat.completions.create(model="llama3.2:3b", messages=[{"role": "user", "content": prompt}], max_tokens=200)
content = resp.choices[0].message.content
print("Chain of thought:\n", content)
print("\n--- Final answer (last line or sentence): ---")
lines = [l.strip() for l in content.strip().split("\n") if l.strip()]
print(lines[-1] if lines else content)


Chain of thought:
 To find out how much 5 pencils cost, we can follow these steps:

Step 1: Determine the cost per pencil.
We know that 3 pencils cost $1.20. To find the cost of one pencil, we divide $1.20 by 3 pencils.

$1.20 ÷ 3 = $0.40

So, each pencil costs $0.40.

Step 2: Determine the cost of 5 pencils.
Now that we know the cost per pencil, we can multiply it by 5 to find out how much 5 pencils cost.

$0.40 × 5 = $2

Therefore, 5 pencils cost $2.

Am I correct?

--- Final answer (last line or sentence): ---
Am I correct?


### 1.3 Self‑Consistency
Self-consistency enhances reasoning accuracy by sampling multiple independent reasoning paths for the same question instead of relying on a single deterministic answer. Each run may follow a slightly different logical chain, and the diversity helps correct individual mistakes. After generating several reasoning traces, you then aggregate the final answers using majority voting.

This approach is especially useful when tasks involve multi-step reasoning or arithmetic, where single-path outputs may be incorrect.

In [6]:
from openai import OpenAI
import re, collections

client = OpenAI(api_key="ollama", base_url="http://localhost:11434/v1")
MODEL = "llama3.2:3b"


def cot_answer(question, temperature=1.2):
    prompt = f"{question}\n\nLet's think step by step."
    resp = client.chat.completions.create(
        model=MODEL, messages=[{"role": "user", "content": prompt}],
        max_tokens=150, temperature=temperature
    )
    content = resp.choices[0].message.content
    # Extract last line or number as final answer
    lines = [l.strip() for l in content.strip().split("\n") if l.strip()]
    last = lines[-1] if lines else content
    # Try to get a single number (e.g. "12" or "12.")
    match = re.search(r"\b(\d+\.?\d*)\s*\.?\s*$", last)
    return (match.group(1) if match else last, content)


def self_consistent(question, n=5):
    answers = []
    traces = []
    for _ in range(n):
        ans, trace = cot_answer(question)
        answers.append(ans)
        traces.append(trace)
    counter = collections.Counter(answers)
    winner = counter.most_common(1)[0][0]
    return winner, counter, traces


question = "What is the square root of 144?"
winner, counter, traces = self_consistent(question, n=5)

print("Votes:", counter)
print("Chosen answer:", winner)

Votes: Counter({'So, √144 = 12!': 1, 'To find the square root of 144, we can start by thinking: "what number multiplied by itself equals 144?"': 1, 'So then you look at each individual factor, find out its square which results in your': 1, '12.': 1, "Answer two that's a decimal sqrt(144) = ± 12 (positive and negative but both just equal 12 since it works for": 1})
Chosen answer: So, √144 = 12!


### 1.4: Sequential Revision

Sequential revision iteratively improves an answer by generating a first draft, critiquing it, and producing revised drafts that condition on prior answers. Each round should be short and focused, so improvements accumulate without drifting from the question.

In [7]:
from openai import OpenAI

client = OpenAI(api_key="ollama", base_url="http://localhost:11434/v1")
MODEL = "llama3.2:3b"


def sequential_revision(question: str, max_steps: int = 3) -> str:
    # Step 1: First draft
    draft = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": f"{question}\n\nProvide a clear, step-by-step answer."}],
        max_tokens=200
    ).choices[0].message.content
    print("Draft 1:", draft[:200] + ("..." if len(draft) > 200 else ""), "\n")

    for step in range(2, max_steps + 1):
        # Step 2: Revise based on previous draft
        rev = client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "user", "content": f"Question: {question}\n\nCurrent answer:\n{draft}\n\nRevise and improve this answer. Keep it concise."}
            ],
            max_tokens=200
        ).choices[0].message.content
        draft = rev
        print(f"Draft {step}:", draft[:200] + ("..." if len(draft) > 200 else ""), "\n")

    return draft


# Run sequential revision and print final output
question = "Explain why the sky is blue in simple terms, then give one sentence on what would change at sunset."
final = sequential_revision(question, max_steps=3)
print("Final answer:", final)

Draft 1: Here's a simple explanation of why the sky is blue:

1. Light from the sun travels through the air in all directions.
2. When light hits tiny molecules of gases in the air, like nitrogen and oxygen, i... 

Draft 2: Here's a revised explanation:

Why is the sky blue?

1. The sun emits different colors of light, each with its own wavelength.
2. When sunlight enters Earth's atmosphere, it encounters tiny molecules ... 

Draft 3: Here's a revised explanation:

Why is the sky blue?

1. The sun emits different colors of light.
2. When sunlight enters Earth's atmosphere, tiny air molecules scatter the light.
3. Blue light scatter... 

Final answer: Here's a revised explanation:

Why is the sky blue?

1. The sun emits different colors of light.
2. When sunlight enters Earth's atmosphere, tiny air molecules scatter the light.
3. Blue light scatters more than other colors, like red and orange.

This scattering makes blue visible from all directions, creating the blue sky.

During sunset

### 1.5 Tree-of-Thoughts

Tree-of-Thoughts (ToT) reframes reasoning as a search problem. Instead of generating one linear chain of thoughts, the model:
1. Generates multiple candidate "thoughts" at each step
2. Evaluates how promising each thought is
3. Expands only the best candidates (beam search)
4. Backtracks if needed

This mirrors how humans solve hard problems: brainstorm options, evaluate them, pursue the best, and backtrack when stuck.

#### Example 1: Word Ladder (Algorithmic ToT)

This example shows ToT as pure beam search without LLM calls. Each "thought" is a candidate word that differs by one letter. We score by edit distance to goal and keep the best candidates.

This demonstrates the **core algorithm** behind ToT: expand, score, prune.

In [8]:
###### Word Ladder Puzzle ##########

def neighbors(word, vocabulary):
    out = []
    for i in range(len(word)):
        for c in "abcdefghijklmnopqrstuvwxyz":
            if c == word[i]:
                continue
            candidate = word[:i] + c + word[i + 1:]
            if candidate in vocabulary:
                out.append(candidate)
    return out


def tree_of_thought(start, goal, vocab, max_depth=5, beam_width=4):
    def edit_dist(a, b):
        return sum(1 for x, y in zip(a, b) if x != y) + abs(len(a) - len(b))

    frontier = [[start]]
    for _ in range(max_depth - 1):
        next_paths = []
        for path in frontier:
            if path[-1] == goal:
                return path
            for nei in neighbors(path[-1], vocab):
                if nei not in path:
                    next_paths.append(path + [nei])
        if not next_paths:
            return None
        next_paths.sort(key=lambda p: edit_dist(p[-1], goal))
        frontier = next_paths[:beam_width]
    for path in frontier:
        if path[-1] == goal:
            return path
    return None


vocab = {"hit","dot","cog","log","dog","lot","lit","hot"}
print(tree_of_thought("hit", "cog", vocab))


['hit', 'hot', 'dot', 'dog', 'cog']


#### Example 2: Generic ToT for Open-Ended Problems

For open-ended problems without verifiable answers, we can still apply ToT by having the LLM both propose and evaluate thoughts.

In [9]:
###### Generic ToT Search ##########

import re
from openai import OpenAI

client = OpenAI(api_key="ollama", base_url="http://localhost:11434/v1")
MODEL = "llama3.2:3b"

def propose_thoughts(question, state, k=2):
    # Propose up to k next “thoughts” that extend the current partial solution/state.
    # Steps: build a short prompt with problem + current state; call your client. Then return a list of stripped strings.
    prompt = f"Problem: {question}\n\nCurrent partial plan:\n{state or '(none)'}\n\nGive exactly {k} distinct next steps or ideas (one per line, short)."
    resp = client.chat.completions.create(model=MODEL, messages=[{"role": "user", "content": prompt}], max_tokens=150)
    content = resp.choices[0].message.content
    lines = [l.strip() for l in content.strip().split("\n") if l.strip()][:k]
    return lines if lines else [content.strip()]


def score_state(question, state):
    prompt = f"Problem: {question}\n\nPartial solution so far:\n{state}\n\nRate how promising this is from 1 to 10 (one integer only)."
    resp = client.chat.completions.create(model=MODEL, messages=[{"role": "user", "content": prompt}], max_tokens=10)
    content = resp.choices[0].message.content
    m = re.search(r"[1-9]|10", content)
    return int(m.group(0)) if m else 5


def tree_of_thoughts(question, depth=2, width=2):
    frontier = [("", 0)]
    for _ in range(depth):
        next_states = []
        for state, _ in frontier:
            for thought in propose_thoughts(question, state, k=width):
                new_state = (state + "\n- " + thought).strip()
                sc = score_state(question, new_state)
                next_states.append((new_state, sc))
        next_states.sort(key=lambda x: -x[1])
        frontier = next_states[:width]
    if not frontier:
        return "", 0
    best = max(frontier, key=lambda x: x[1])
    return best[0], best[1]


question = "Design a plan for a weekend science workshop for 12-year-olds."
solution, score = tree_of_thoughts(question)

print(f"Best solution (score {score}):\n{solution}")

Best solution (score 9):
- Here are two potential next steps to help design the plan:
- - Step 1: Determine the age group and interests of the participants by surveying parents/guardians or gathering information about the children's past experiences with science. This will help tailor the activities to their level of understanding and enthusiasm.


---  
# 2‑ Training Models for Reasoning

### 2.1: CoT Training
Chain-of-Thought (CoT) training conditions the model on explicit rationales during fine-tuning. Instead of teaching the model to output only the final answer, we train on (question, rationale, answer) so the model learns to internalize multi-step reasoning patterns. A practical recipe is STaR (Self-Taught Reasoner), which uses a stronger teacher model to bootstrap rationales that a smaller student can learn from.

For tasks that require multi-hop reasoning, models fine-tuned on rationales often achieve higher accuracy and are more stable at inference time than models trained on direct answers only. 

Training a full language model is beyond the scope of this notebook, but here is the high-level workflow followed by a short pseudocode:
- Collect questions: Prepare a dataset of questions and correct answers.
- Generate rationales: Use a strong LLM to produce step-by-step reasoning ending with the correct answer.
- Filter and clean: Discard incorrect or low-quality rationales.
- Prepare training data: Format triples (question, rationale, answer) for supervised fine-tuning.
- Fine-tune: Fine-tune the LLM on rationales.
- Iterate: Refine prompts, improve data quality, and retrain for stronger reasoning.

In [ ]:
# Pseudocode (STaR loop)
# for round in 1 ... iters:
    # STEP 1: self-generate reasoning (teacher creates rationale + answer)
    # STEP 2: keep only correct, high-quality traces
    # STEP 3: fine-tune student on (question, rationale, answer) data

### 2.2: ORM vs PRM + RL
Training a Reward Model (RM) allows large language models to be improved through reinforcement learning (RL). Instead of fine-tuning directly on examples, we train a separate model that can score or rank model outputs, and use those scores as feedback signals to refine the policy model.

Two main reward modeling approaches are ORM (predicts a scalar reward for the final answer) and PRM (evaluates the reasoning steps instead of just the outcome)



| Approach | Typical loss | When to use |
|-----------|-------------|-------------|
|*Outcome Reward Model* | Predict scalar reward | Easy to collect training data using verifiers |
|*Process Reward Model* | Predict rewards per step | Difficult to collect training data but more accurate |
| *RLHF* | Use RM as reward in **RL** fine‑tuning | Aligns policy with human signals | Aligns model policy with human or synthetic preferences




In [ ]:
# for round = 1 ... iters:
    # STEP 1:  Generate reasoning
        # sample a minibatch of questions
        # policy roll‑out (actions + log‑probs)
    # STEP 2:  Score the trajectory
        # ORM: scalar reward for the final answer / PRM: scalar reward for the thought process
    # STEP 3:  Reinforce the policy (PPO)

### 2.3 Inspect a reasoning model

Now that we've discussed how reasoning models are trained, let's see one in action. We'll use **DeepSeek-R1**, a reasoning model that produces explicit *thinking tokens* before giving its final answer. The model wraps its internal chain-of-thought inside `<think>...</think>` tags, followed by a clean final response.

In the cell below we send a question to DeepSeek-R1 and parse the output to separate:
- **Thinking tokens** — the model's internal reasoning process (hidden from the end user in production).
- **Final answer** — the polished response the user actually sees.

We use `deepseek-r1:1.5b` here for speed. You can switch to `deepseek-r1:8b` for higher-quality reasoning, but it will take longer to run. Pull whichever variant you want to try:

In [13]:
import re
try:
    from openai import OpenAI
except ModuleNotFoundError as e:
    print("Python module not found:", e)
    print("Install it with: pip install openai")
    raise

# Step 1: Client and DeepSeek-R1 model
client = OpenAI(api_key="ollama", base_url="http://localhost:11434/v1")
MODEL = "deepseek-r1:1.5b"

# Step 2: Math question
question = "A store sells apples for $2 each and oranges for $3 each. If I buy 4 apples and 2 oranges, how much do I pay?"

# Step 3: Call model
content = None
try:
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": question}],
        max_tokens=500
    )
    content = resp.choices[0].message.content
except Exception as e:
    if "404" in str(e) or "not found" in str(e).lower():
        print("Ollama model not found (the AI model name, not a Python module).")
        print("1. Start Ollama in another terminal: ollama serve")
        print("2. Pull the model: ollama pull deepseek-r1:1.5b")
        print("   Or: ollama pull deepseek-r1:8b")
        content = None
    else:
        raise

if content is not None:
    # Step 4: Separate <think>...</think> and final answer
    think_match = re.search(r"<think>(.*?)</think>", content, re.DOTALL)
    thinking = think_match.group(1).strip() if think_match else "(no think block found)"
    final = re.sub(r"<think>.*?</think>", "", content, flags=re.DOTALL).strip() if think_match else content

    print("=== Thinking (internal reasoning) ===")
    print(thinking[:500] + ("..." if len(thinking) > 500 else ""))
    print("\n=== Final answer (user-facing) ===")
    print(final)

=== Thinking (internal reasoning) ===
(no think block found)

=== Final answer (user-facing) ===
**Solution:**

Let's determine the total cost step by step.

**Step 1: Identify the individual prices.**
- **Cost of apples:** \$2 each
- **Cost of oranges:** \$3 each

**Step 2: Determine the quantities purchased.**
- You buy:
  - **4 apples**
  - **2 oranges**

**Step 3: Calculate the total cost for apples.**
\[
\$2 \text{ per apple} \times 4 \text{ apples} = \$8
\]

**Step 4: Calculate the total cost for oranges.**
\[
\$3 \text{ per orange} \times 2 \text{ oranges} = \$6
\]

**Step 5: Add the two amounts together to find the overall total.**
\[
\$8 + \$6 = \$14
\]

**Final Answer:**
\[
\boxed{\$14}
\]


---  
# 3‑ A Deep Research Agent

A deep-research agent pairs a reasoning model with external tools for web search and retrieval. We will follow the ReAct pattern: the model writes short thoughts, decides when to call tools, reads observations, and continues reasoning until it can answer or reaches a step limit.

We now combine a **search tool** with an LLM in a multi-step setup. We follow the *ReAct* pattern (reason → tool → observation):

1. The model reasons and decides to use tools
2. The agent searches and feeds condensed snippets back as context
3. Iterate until the model answers or hits a step limit

We use `create_agent` from `langchain.agents`, which builds a ReAct-style agent graph. Note: the agent model must support **tool calling** (e.g., `llama3.2:3b`). Models like `deepseek-r1` are reasoning models that do not support native tool calling and cannot be used directly as the agent LLM. We can stick to the `llama3.2:3b` or `qwen2.5:3b-instruct` for this section.

In [17]:
from ddgs import DDGS
from langchain_core.tools import tool


@tool
def ddg_search(query: str, k: int = 5) -> str:
    """Run a web search and return snippets. Use for finding current information."""
    results = list(DDGS().text(query, max_results=k))
    return "\n".join(r.get("body", r.get("title", "")) for r in results)

In [16]:
from langchain.agents import create_agent
from langchain_ollama import ChatOllama

MODEL = "llama3.2:3b"  # or "qwen2.5:3b-instruct" if you run: ollama pull qwen2.5:3b-instruct
question = "What are the best resources to learn machine learning in 2025?"

# Step 1: Initialize the LLM via ChatOllama (must support tool calling)
llm = ChatOllama(model=MODEL, temperature=0)

# Step 2: Build a tool-calling agent with DuckDuckGo search
agent = create_agent(llm, [ddg_search])

# Step 3: Ask a query and let the agent search + reason to produce an answer
out = agent.invoke({"messages": [{"role": "user", "content": question}]})
print(out["messages"][-1].content)

Based on the tool call response, here's a formatted answer to the original user question:

**Best Resources to Learn Machine Learning in 2025**

To get started with machine learning, it's essential to have a solid foundation in necessary math concepts such as linear algebra, probability, and calculus. Additionally, having experience with programming languages like Python or R can be beneficial.

Here are some top resources to learn machine learning:

1. **Online Courses**:
	* Coursera: Machine Learning by Andrew Ng
	* edX: Machine Learning by Microsoft
	* Stanford University's Machine Learning Course
2. **Books**:
	* "Pattern Recognition and Machine Learning" by Christopher Bishop
	* "Deep Learning" by Ian Goodfellow, Yoshua Bengio, and Aaron Courville
3. **YouTube Playlists**:
	* 3Blue1Brown (Grant Sanderson) - Machine Learning Explained
	* Machine Learning Mastery (Jason Brownlee)
4. **Websites**:
	* Kaggle: A platform for machine learning competitions and hosting datasets
	* Scikit-

# 4- (Optional) Multi-Agent Deep Research

Instead of a single agent, we can design multiple collaborating agents that work in parallel:

1. **Planner**: Analyzes the query and breaks it into sub-questions
2. **Researchers**: Run in parallel, each searching and summarizing findings for one sub-question  
3. **Synthesizer**: Combines all research into a coherent final report

This setup improves coverage and speed by parallelizing the research phase.

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from openai import OpenAI
from ddgs import DDGS

client = OpenAI(api_key="ollama", base_url="http://localhost:11434/v1")
MODEL = "llama3.2:3b"


def plan_research(query: str) -> list[str]:
    """Planner agent: breaks query into sub-questions."""
    prompt = f"""Break this research question into 1-5 focused sub-questions. Return only the sub-questions, one per line, no numbering.

Question: {query}"""
    resp = client.chat.completions.create(model=MODEL, messages=[{"role": "user", "content": prompt}], max_tokens=200)
    content = resp.choices[0].message.content
    return [l.strip() for l in content.strip().split("\n") if l.strip()]


def search_and_summarize(sub_question: str) -> dict:
    """Researcher agent: searches web and summarizes findings for one sub-question."""
    results = list(DDGS().text(sub_question, max_results=3))
    snippets = "\n".join(r.get("body", r.get("title", "")) for r in results)
    prompt = f"Sub-question: {sub_question}\n\nSearch results:\n{snippets}\n\nWrite a concise summary (2-4 sentences) answering the sub-question."
    resp = client.chat.completions.create(model=MODEL, messages=[{"role": "user", "content": prompt}], max_tokens=150)
    summary = resp.choices[0].message.content
    return {"question": sub_question, "summary": summary}


def synthesize_report(query: str, findings: list[dict]) -> str:
    """Synthesizer agent: combines all findings into a coherent report."""
    block = "\n\n".join(f"### {f['question']}\n{f['summary']}" for f in findings)
    prompt = f"Original question: {query}\n\nFindings:\n{block}\n\nWrite a well-structured markdown report that answers the original question. Use the findings above."
    resp = client.chat.completions.create(model=MODEL, messages=[{"role": "user", "content": prompt}], max_tokens=400)
    return resp.choices[0].message.content


def deep_research(query: str) -> str:
    """Run the full multi-agent deep research pipeline."""
    sub_questions = plan_research(query)
    print("Sub-questions:", sub_questions)
    with ThreadPoolExecutor(max_workers=len(sub_questions)) as ex:
        findings = list(ex.map(search_and_summarize, sub_questions))
    report = synthesize_report(query, findings)
    return report


# Run the multi-agent research
query = "What are the best resources to learn machine learning in 2025?"
report = deep_research(query)
print("=" * 60)
print("FINAL REPORT")
print("=" * 60)
print(report)

Sub-questions: ['What are some top online courses and tutorials for machine learning beginners?', 'How can machine learning practitioners stay up-to-date with the latest advancements in the field?', 'What is the role of real-world projects and applications in learning machine learning, and how can they be utilized effectively?', 'Do online communities, forums, and social media platforms contribute to a successful machine learning learning process?', "How do different educational programs (degrees, boot camps, self-study) impact an individual's ability to learn machine learning?"]
FINAL REPORT
# Mastering Machine Learning in 2025: Top Resources and Strategies for Success

## Introduction

Machine learning (ML) has revolutionized various industries, and its applications continue to grow exponentially. However, acquiring ML skills can be a daunting task, especially for beginners. This report highlights the most effective resources and strategies for learning machine learning in 2025.

## 

## 🎉 Congratulations!

You have:
* Practiced various inference-time reasoning methods (CoT, self-consistency, sequential revision, ToT)
* Gained intuition about training reasoning models (STaR, ORM/PRM)
* Built a **deep-research agent** with tool calling and ReAct-style reasoning
* Implemented a **multi-agent system** with parallel research and report synthesis


👏 **Great job!** Take a moment to celebrate. The techniques you implemented here power many production agents and chatbots.